In [1]:
!pip install --upgrade pip

  Using cached pip-26.2.1-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.2.1-py3-none-any.whl (1.8 MB)



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\Deno\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip
ERROR: To modify pip, please run the following command:
C:\Users\Deno\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [2]:
!pip install --index-url https://pypi.org/simple imbalanced-learn


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\Deno\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [3]:
!pip install imbalamced-learn

ERROR: Could not find a version that satisfies the requirement imbalamced-learn (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\Deno\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for imbalamced-learn


In [ ]:

import os
import warnings

import joblib
import matplotlib
matplotlib.use("Agg")  # works without a display; remove if you want pop-up windows
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline  # NOT sklearn.pipeline.Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_predict,
    train_test_split,
)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------------
# CONFIGURATION
# ----------------------------------------------------------------------------
DATA_PATH = "cleaned_ml_ready_dataset.csv"   # change to "creditcard.csv" for the Kaggle data
OUTPUT_DIR = "project2_outputs"
RANDOM_STATE = 42
TEST_SIZE = 0.20
CV_FOLDS = 5
N_JOBS = -1

# TARGET SETUP
# - If the file has a binary "Class" column (Kaggle creditcard.csv, 0 = legit,
#   1 = fraud) that column is used automatically.
# - Otherwise (the provided cleaned_ml_ready_dataset.csv has no fraud label) the
#   positive "anomaly" class is built from OrderStatus == POSITIVE_STATUS.
STATUS_COLUMN = "OrderStatus"
POSITIVE_STATUS = "Returned"   # minority "bad outcome" class, ~20.6% of orders

os.makedirs(OUTPUT_DIR, exist_ok=True)
sns.set_theme(style="whitegrid")


def banner(text):
    print("\n" + "=" * 78 + f"\n{text}\n" + "=" * 78)


# ----------------------------------------------------------------------------
# 1. LOAD DATA AND BUILD TARGET
# ----------------------------------------------------------------------------
banner("1. LOAD DATA")
df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
print(f"Missing values: {int(df.isna().sum().sum())} | Duplicate rows: {int(df.duplicated().sum())}")

if "Class" in df.columns:
    target_name = "Class"
    y = df["Class"].astype(int)
    X = df.drop(columns=["Class"])
    print("Using existing 'Class' column as target (1 = fraud).")
elif STATUS_COLUMN in df.columns:
    target_name = f"is_{POSITIVE_STATUS.lower()}"
    y = (df[STATUS_COLUMN] == POSITIVE_STATUS).astype(int)
    # The status column IS the label source, so it must never be a feature.
    X = df.drop(columns=[STATUS_COLUMN])
    print(f"Target built from {STATUS_COLUMN} == '{POSITIVE_STATUS}' (1 = positive/anomaly).")
else:
    raise ValueError("No 'Class' or status column found - set the target manually.")

# Keep only numeric features (the cleaned dataset is already one-hot encoded)
X = X.select_dtypes(include=[np.number])
feature_names = X.columns.tolist()
print(f"Features used: {X.shape[1]}")

# ----------------------------------------------------------------------------
# 2. CLASS IMBALANCE + THE ACCURACY ILLUSION
# ----------------------------------------------------------------------------
banner("2. CLASS IMBALANCE AND THE ACCURACY ILLUSION")
counts = y.value_counts().sort_index()
rates = y.value_counts(normalize=True).sort_index() * 100
print(pd.DataFrame({"count": counts, "percent": rates.round(2)}))
print(f"Imbalance ratio (majority:minority) = {counts.max() / counts.min():.1f} : 1")

fig, ax = plt.subplots(figsize=(5, 4))
sns.barplot(x=counts.index.astype(str), y=counts.values, palette=["#3b8ea5", "#f26b5b"], ax=ax)
for i, v in enumerate(counts.values):
    ax.text(i, v, f"{v}\n({rates.values[i]:.2f}%)", ha="center", va="bottom")
ax.set_title("Class distribution")
ax.set_xlabel(f"{target_name} (1 = positive)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/01_class_distribution.png", dpi=150)
plt.close()

# ----------------------------------------------------------------------------
# 3. STRATIFIED TRAIN / TEST SPLIT  (BEFORE scaling or SMOTE -> no leakage)
# ----------------------------------------------------------------------------
banner("3. STRATIFIED TRAIN/TEST SPLIT")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape} | positives: {y_train.mean() * 100:.2f}%")
print(f"Test : {X_test.shape} | positives: {y_test.mean() * 100:.2f}%")

# Demonstrate the accuracy trap with a model that always predicts the majority
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
d_pred = dummy.predict(X_test)
print("\nDummy 'always legitimate' model on the test set:")
print(f"  Accuracy  : {(d_pred == y_test).mean():.4f}   <-- looks great, is useless")
print(f"  Precision : {precision_score(y_test, d_pred, zero_division=0):.4f}")
print(f"  Recall    : {recall_score(y_test, d_pred, zero_division=0):.4f}")
print(f"  ROC-AUC   : {roc_auc_score(y_test, dummy.predict_proba(X_test)[:, 1]):.4f}")

# ----------------------------------------------------------------------------
# 4. LEAK-FREE PIPELINES (imblearn Pipeline applies SMOTE only when fitting)
# ----------------------------------------------------------------------------
banner("4. BUILD PIPELINES")
lr_pipe = ImbPipeline(steps=[
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("classifier", LogisticRegression(max_iter=2000, solver="lbfgs", random_state=RANDOM_STATE)),
])

rf_pipe = ImbPipeline(steps=[
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("classifier", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=N_JOBS)),
])
print("Logistic Regression:", [n for n, _ in lr_pipe.steps])
print("Random Forest      :", [n for n, _ in rf_pipe.steps])

# ----------------------------------------------------------------------------
# 5. HYPERPARAMETER TUNING WITH GridSearchCV (SMOTE inside every fold)
# ----------------------------------------------------------------------------
banner("5. GRIDSEARCHCV TUNING")
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

lr_grid = {
    "smote__k_neighbors": [3, 5, 7],
    "classifier__C": [0.01, 0.1, 1.0],
}
rf_grid = {
    "smote__k_neighbors": [3, 5, 7],
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [10, 20, None],
}

# Several metrics are tracked; the model is refit on ROC-AUC (accuracy is NOT tracked)
scoring = {
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
    "recall": "recall",
    "precision": "precision",
    "f1": "f1",
}

searches = {}
for name, pipe, grid in [
    ("Logistic Regression", lr_pipe, lr_grid),
    ("Random Forest", rf_pipe, rf_grid),
]:
    print(f"\nTuning {name} ...")
    gs = GridSearchCV(
        estimator=pipe,
        param_grid=grid,
        scoring=scoring,
        refit="roc_auc",
        cv=cv,
        n_jobs=N_JOBS,
        verbose=0,
        return_train_score=False,
    )
    gs.fit(X_train, y_train)
    searches[name] = gs
    best = gs.best_index_
    print(f"  Best params      : {gs.best_params_}")
    print(f"  CV ROC-AUC       : {gs.cv_results_['mean_test_roc_auc'][best]:.4f}")
    print(f"  CV PR-AUC        : {gs.cv_results_['mean_test_average_precision'][best]:.4f}")
    print(f"  CV Recall        : {gs.cv_results_['mean_test_recall'][best]:.4f}")
    print(f"  CV Precision     : {gs.cv_results_['mean_test_precision'][best]:.4f}")
    print(f"  CV F1            : {gs.cv_results_['mean_test_f1'][best]:.4f}")
    pd.DataFrame(gs.cv_results_).sort_values("rank_test_roc_auc").to_csv(
        f"{OUTPUT_DIR}/cv_results_{name.replace(' ', '_').lower()}.csv", index=False
    )

# ----------------------------------------------------------------------------
# 6. FINAL EVALUATION ON THE UNTOUCHED TEST SET
# ----------------------------------------------------------------------------
banner("6. TEST SET EVALUATION (default 0.5 threshold)")
rows = []
test_probs = {}
for name, gs in searches.items():
    model = gs.best_estimator_
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    test_probs[name] = proba
    rows.append({
        "Model": name,
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, proba),
        "PR-AUC": average_precision_score(y_test, proba),
    })
    print(f"\n--- {name} ---")
    print(f"Best params: {gs.best_params_}")
    rep = classification_report(y_test, pred, target_names=["Negative (0)", "Positive (1)"],
                                zero_division=0, output_dict=True)
    # Accuracy row intentionally dropped - the project forbids relying on it
    print(pd.DataFrame(rep).T.drop(index=["accuracy"], errors="ignore").round(3))
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    print(f"Confusion matrix -> TN={tn} FP={fp} FN={fn} TP={tp}")

results = pd.DataFrame(rows).set_index("Model").round(4)
banner("MODEL COMPARISON (test set)")
print(results)
results.to_csv(f"{OUTPUT_DIR}/test_metrics.csv")

# Reference line: the positive-class rate is the PR-AUC of a random model
print(f"\nPR-AUC of a random classifier = positive rate = {y_test.mean():.4f}")
print("ROC-AUC of a random classifier = 0.5000")

# ----------------------------------------------------------------------------
# 7. ABLATION: DOES SMOTE ACTUALLY HELP? (Random Forest with vs without)
# ----------------------------------------------------------------------------
banner("7. ABLATION: RANDOM FOREST WITH vs WITHOUT SMOTE")
best_rf_params = {k: v for k, v in searches["Random Forest"].best_params_.items()
                  if k.startswith("classifier__")}
rf_no_smote = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=N_JOBS)
rf_no_smote.set_params(**{k.replace("classifier__", ""): v for k, v in best_rf_params.items()})
rf_no_smote.fit(X_train, y_train)
p_ns = rf_no_smote.predict_proba(X_test)[:, 1]
pred_ns = (p_ns >= 0.5).astype(int)
ablation = pd.DataFrame({
    "RF + SMOTE": results.loc["Random Forest"],
    "RF (no SMOTE)": pd.Series({
        "Precision": precision_score(y_test, pred_ns, zero_division=0),
        "Recall": recall_score(y_test, pred_ns, zero_division=0),
        "F1": f1_score(y_test, pred_ns, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, p_ns),
        "PR-AUC": average_precision_score(y_test, p_ns),
    }),
}).round(4)
print(ablation)
ablation.to_csv(f"{OUTPUT_DIR}/ablation_smote.csv")

# ----------------------------------------------------------------------------
# 8. THRESHOLD TUNING (choose threshold on TRAIN out-of-fold predictions only)
# ----------------------------------------------------------------------------
banner("8. DECISION-THRESHOLD TUNING (best model by CV ROC-AUC)")
# Model selection uses CROSS-VALIDATION scores only, never the test set
best_name = max(searches, key=lambda n: searches[n].best_score_)
best_model = searches[best_name].best_estimator_
print(f"Best model by CV ROC-AUC: {best_name}")

oof = cross_val_predict(best_model, X_train, y_train, cv=cv, method="predict_proba", n_jobs=N_JOBS)[:, 1]
prec, rec, thr = precision_recall_curve(y_train, oof)
f1s = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-12, None)
best_thr = float(thr[np.argmax(f1s)])
print(f"Threshold maximising out-of-fold F1: {best_thr:.3f}")

proba_best = test_probs[best_name]
pred_thr = (proba_best >= best_thr).astype(int)
print(f"\nTest metrics at tuned threshold {best_thr:.3f}:")
print(f"  Precision: {precision_score(y_test, pred_thr, zero_division=0):.4f}")
print(f"  Recall   : {recall_score(y_test, pred_thr, zero_division=0):.4f}")
print(f"  F1       : {f1_score(y_test, pred_thr, zero_division=0):.4f}")
print(f"  Confusion matrix:\n{confusion_matrix(y_test, pred_thr)}")

# ----------------------------------------------------------------------------
# 9. VISUALS
# ----------------------------------------------------------------------------
banner("9. SAVING FIGURES")
# 9a Confusion matrices
fig, axes = plt.subplots(1, len(searches), figsize=(5.5 * len(searches), 4.5))
for ax, (name, gs) in zip(np.atleast_1d(axes), searches.items()):
    pred = (test_probs[name] >= 0.5).astype(int)
    ConfusionMatrixDisplay(confusion_matrix(y_test, pred),
                           display_labels=["Negative", "Positive"]).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"{name}\n(threshold 0.5)")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/02_confusion_matrices.png", dpi=150)
plt.close()

# 9b ROC curves
fig, ax = plt.subplots(figsize=(6, 5))
for name in searches:
    RocCurveDisplay.from_predictions(y_test, test_probs[name], name=name, ax=ax)
ax.plot([0, 1], [0, 1], "k--", label="Random (AUC = 0.50)")
ax.set_title("ROC curves (test set)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/03_roc_curves.png", dpi=150)
plt.close()

# 9c Precision-Recall curves
fig, ax = plt.subplots(figsize=(6, 5))
for name in searches:
    PrecisionRecallDisplay.from_predictions(y_test, test_probs[name], name=name, ax=ax)
ax.axhline(y_test.mean(), color="k", ls="--", label=f"Random (precision = {y_test.mean():.2f})")
ax.set_title("Precision-Recall curves (test set)")
ax.legend(loc="upper right")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/04_pr_curves.png", dpi=150)
plt.close()

# 9d Metric comparison
ax = results.plot(kind="bar", figsize=(8, 4.5), rot=0, colormap="viridis")
ax.set_title("Test-set metrics by model (no accuracy)")
ax.set_ylim(0, 1)
ax.legend(loc="upper right", ncol=3, fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/05_metric_comparison.png", dpi=150)
plt.close()

# 9e Random Forest feature importance
rf_best = searches["Random Forest"].best_estimator_.named_steps["classifier"]
imp = pd.Series(rf_best.feature_importances_, index=feature_names).sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(7, 5.5))
sns.barplot(x=imp.values, y=imp.index, color="#3b8ea5", ax=ax)
ax.set_title("Random Forest - top 15 feature importances")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/06_rf_feature_importance.png", dpi=150)
plt.close()

# 9f Logistic Regression coefficients (interpretability)
lr_best = searches["Logistic Regression"].best_estimator_.named_steps["classifier"]
coefs = pd.Series(lr_best.coef_[0], index=feature_names)
top = coefs.reindex(coefs.abs().sort_values(ascending=False).head(15).index)
fig, ax = plt.subplots(figsize=(7, 5.5))
sns.barplot(x=top.values, y=top.index, palette=["#f26b5b" if v > 0 else "#3b8ea5" for v in top.values], ax=ax)
ax.set_title("Logistic Regression - top 15 coefficients (standardised features)")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/07_lr_coefficients.png", dpi=150)
plt.close()
print(f"Figures saved to ./{OUTPUT_DIR}/")

# ----------------------------------------------------------------------------
# 10. SAVE THE BEST MODEL
# ----------------------------------------------------------------------------
banner("10. SAVE MODEL")
joblib.dump({"model": best_model, "threshold": best_thr, "features": feature_names},
            f"{OUTPUT_DIR}/best_fraud_model.joblib")
print(f"Saved: {OUTPUT_DIR}/best_fraud_model.joblib  (model: {best_name}, threshold: {best_thr:.3f})")

# ----------------------------------------------------------------------------
# 11. SUMMARY
# ----------------------------------------------------------------------------
banner("PROJECT SUMMARY")
print(f"Samples: {len(df)} | Features: {len(feature_names)} | Positive rate: {y.mean() * 100:.2f}%")
print("Leakage controls: split first; StandardScaler + SMOTE live inside imblearn Pipeline;")
print("                  SMOTE applied only to training folds; test set untouched until the end.")
print("Metrics reported: Precision, Recall, F1, ROC-AUC, PR-AUC (accuracy discarded).")
print(f"Best model: {best_name}")
print(results)
print(f"\nAll outputs written to ./{OUTPUT_DIR}/")


1. LOAD DATA
Shape: (1200, 40)
Missing values: 0 | Duplicate rows: 0
Target built from OrderStatus == 'Returned' (1 = positive/anomaly).
Features used: 39

2. CLASS IMBALANCE AND THE ACCURACY ILLUSION
             count  percent
OrderStatus                
0              953    79.42
1              247    20.58
Imbalance ratio (majority:minority) = 3.9 : 1

3. STRATIFIED TRAIN/TEST SPLIT
Train: (960, 39) | positives: 20.62%
Test : (240, 39) | positives: 20.42%

Dummy 'always legitimate' model on the test set:
  Accuracy  : 0.7958   <-- looks great, is useless
  Precision : 0.0000
  Recall    : 0.0000
  ROC-AUC   : 0.5000

4. BUILD PIPELINES
Logistic Regression: ['scaler', 'smote', 'classifier']
Random Forest      : ['smote', 'classifier']

5. GRIDSEARCHCV TUNING

Tuning Logistic Regression ...


  File "C:\Users\Deno\AppData\Local\Programs\Python\Python314\Lib\site-packages\joblib\externals\loky\backend\context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


  Best params      : {'classifier__C': 0.01, 'smote__k_neighbors': 5}
  CV ROC-AUC       : 0.5426
  CV PR-AUC        : 0.2555
  CV Recall        : 0.4844
  CV Precision     : 0.2338
  CV F1            : 0.3151

Tuning Random Forest ...
